# 01 — Exploratory Data Analysis
### CKD Prediction Project · MSc AI Portfolio

**Dataset:** Chronic Kidney Disease · 1,659 patients · 51 features · No missing values  
**Target:** `Diagnosis` — 1 = CKD, 0 = No CKD  

**Notebook goals:**
1. Understand the dataset structure and feature types
2. Examine the class imbalance
3. Explore distributions of the 10 selected features vs diagnosis
4. Look at correlations
5. Identify any patterns that justify our feature selection decisions


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Selected features (chosen by permutation importance — see 02_feature_selection.py)
SELECTED_FEATURES = [
    'SerumCreatinine', 'GFR', 'MuscleCramps', 'FastingBloodSugar',
    'ProteinInUrine', 'Itching', 'SerumElectrolytesSodium',
    'HemoglobinLevels', 'NauseaVomiting', 'PhysicalActivity'
]

print('Libraries loaded.')

## 2. Load Dataset

In [ ]:
df_raw = pd.read_csv('../data/ckd_data.csv')
df = df_raw.drop(columns=['PatientID', 'DoctorInCharge'], errors='ignore')

print(f'Shape: {df.shape}')
print(f'Columns: {df.shape[1] - 1} features + 1 target')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head(3)

## 3. Dataset Overview

In [ ]:
# Feature type breakdown
dtype_counts = df.drop(columns=['Diagnosis']).dtypes.value_counts()
print('Feature dtypes:')
print(dtype_counts)
print(f'\nTotal features: {len(df.columns) - 1}')
print(f'Total patients: {len(df)}')

In [ ]:
# Quick stats on selected features
df[SELECTED_FEATURES + ['Diagnosis']].describe().round(2)

## 4. Class Imbalance

This is the most important thing to understand about this dataset before doing anything else.  
A naive model that predicts *CKD for everyone* would score **91.9% accuracy** — so accuracy is a useless metric here.


In [ ]:
counts = df['Diagnosis'].value_counts()
labels = ['CKD (1)', 'No CKD (0)']
values = [counts[1], counts[0]]
pcts   = [counts[1]/len(df)*100, counts[0]/len(df)*100]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart
bars = axes[0].bar(labels, values, color=['#ef4444', '#10b981'], width=0.45, edgecolor='white')
for bar, pct in zip(bars, pcts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                 f'{int(bar.get_height())}\n({pct:.1f}%)', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_ylabel('Number of Patients')
axes[0].set_ylim([0, 1800])

# Pie chart
axes[1].pie(values, labels=labels, colors=['#ef4444', '#10b981'],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportion', fontweight='bold')

fig.suptitle(f'Imbalance Ratio: {counts[1]/counts[0]:.1f}:1  —  Oversampling applied during training',
             fontsize=11, color='#64748b')
plt.tight_layout()
plt.show()

print(f'CKD patients    : {counts[1]} ({pcts[0]:.1f}%)')
print(f'No CKD patients : {counts[0]} ({pcts[1]:.1f}%)')
print(f'Imbalance ratio : {counts[1]/counts[0]:.1f}x')
print(f'\nDummy classifier accuracy (always predict CKD): {counts[1]/len(df)*100:.1f}%')

## 5. Selected Feature Distributions by Class

For each of the 10 selected features, we compare the distribution for CKD vs No CKD patients.  
Well-separated distributions = strong predictors.


In [ ]:
ckd    = df[df['Diagnosis'] == 1]
no_ckd = df[df['Diagnosis'] == 0]

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(SELECTED_FEATURES):
    ax = axes[i]
    ax.hist(no_ckd[col], bins=30, alpha=0.65, color='#10b981', label='No CKD', density=True)
    ax.hist(ckd[col],    bins=30, alpha=0.65, color='#ef4444', label='CKD',    density=True)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.set_yticks([])
    ax.legend(fontsize=7)
    # add means as vertical lines
    ax.axvline(no_ckd[col].mean(), color='#059669', linestyle='--', linewidth=1.2)
    ax.axvline(ckd[col].mean(),    color='#dc2626', linestyle='--', linewidth=1.2)

fig.suptitle('Feature Distributions by Class  (dashed = group mean)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Box Plots — Median and Spread per Class

Box plots are easier to read for spotting the *median shift* between classes.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for i, col in enumerate(SELECTED_FEATURES):
    ax = axes[i]
    data_to_plot = [no_ckd[col].values, ckd[col].values]
    bp = ax.boxplot(data_to_plot, patch_artist=True, widths=0.5,
                    medianprops={'color': 'white', 'linewidth': 2})
    bp['boxes'][0].set_facecolor('#10b981')
    bp['boxes'][1].set_facecolor('#ef4444')
    for element in ['whiskers', 'caps', 'fliers']:
        for patch in bp[element]:
            patch.set_color('#94a3b8')
    ax.set_xticklabels(['No CKD', 'CKD'], fontsize=9)
    ax.set_title(col, fontsize=10, fontweight='bold')

fig.suptitle('Box Plots — Selected Features by Class', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Deep Dive — Key Biomarkers

**SerumCreatinine** and **GFR** are the two most predictive features (permutation importance rank 1 & 2).  
Clinically, GFR < 60 mL/min is the standard threshold for CKD diagnosis.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Scatter: GFR vs SerumCreatinine coloured by Diagnosis
colors_map = {1: '#ef4444', 0: '#10b981'}
colors_arr = df['Diagnosis'].map(colors_map)
axes[0].scatter(df['GFR'], df['SerumCreatinine'], c=colors_arr,
                alpha=0.4, s=15, edgecolors='none')
axes[0].axvline(60, color='#f59e0b', linestyle='--', linewidth=1.5, label='GFR=60 (CKD threshold)')
axes[0].set_xlabel('GFR (mL/min)')
axes[0].set_ylabel('Serum Creatinine (mg/dL)')
axes[0].set_title('GFR vs Serum Creatinine', fontweight='bold')
axes[0].legend(fontsize=8)
# manual legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#ef4444', label='CKD'), Patch(facecolor='#10b981', label='No CKD')]
axes[0].legend(handles=legend_elements + [plt.Line2D([0],[0], color='#f59e0b', linestyle='--', label='GFR=60')],
               fontsize=8)

# GFR distribution with clinical threshold
axes[1].hist(no_ckd['GFR'], bins=30, alpha=0.7, color='#10b981', label='No CKD', density=True)
axes[1].hist(ckd['GFR'],    bins=30, alpha=0.7, color='#ef4444', label='CKD',    density=True)
axes[1].axvline(60, color='#f59e0b', linestyle='--', linewidth=2, label='GFR=60 threshold')
axes[1].set_xlabel('GFR (mL/min)')
axes[1].set_title('GFR Distribution by Class', fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].set_yticks([])

# How often GFR < 60 correctly predicts CKD
gfr_low  = df[df['GFR'] < 60]
gfr_high = df[df['GFR'] >= 60]
categories = ['GFR < 60', 'GFR ≥ 60']
ckd_pct    = [gfr_low['Diagnosis'].mean()*100, gfr_high['Diagnosis'].mean()*100]
bars = axes[2].bar(categories, ckd_pct, color=['#ef4444', '#3b82f6'], width=0.45, edgecolor='white')
axes[2].bar_label(bars, fmt='%.1f%%', padding=4, fontsize=11, fontweight='bold')
axes[2].set_ylabel('% Diagnosed as CKD')
axes[2].set_title('CKD Rate by GFR Range', fontweight='bold')
axes[2].set_ylim([0, 115])
axes[2].axhline(df['Diagnosis'].mean()*100, linestyle='--', color='gray', linewidth=1, label='Overall rate')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f'GFR < 60 → CKD rate: {gfr_low["Diagnosis"].mean()*100:.1f}% (n={len(gfr_low)})')
print(f'GFR ≥ 60 → CKD rate: {gfr_high["Diagnosis"].mean()*100:.1f}% (n={len(gfr_high)})')

## 8. Correlation Heatmap — Selected Features

Checking for multicollinearity. Highly correlated features carry redundant information.


In [ ]:
corr = df[SELECTED_FEATURES].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle only
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, annot_kws={'size': 9},
            square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — 10 Selected Features', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Flag any highly correlated pairs
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        val = corr.iloc[i, j]
        if abs(val) > 0.5:
            high_corr.append((corr.columns[i], corr.columns[j], val))

if high_corr:
    print('Pairs with |correlation| > 0.5:')
    for a, b, v in high_corr:
        print(f'  {a} & {b}: {v:.3f}')
else:
    print('No pairs with |correlation| > 0.5 — no multicollinearity concerns.')

## 9. Feature-Target Correlation

Point-biserial correlation of each selected feature with the binary target.  
Note: even the top features only reach ~0.20 — this dataset has weak individual signals,  
which is why the model relies on combining all 10.


In [ ]:
target_corr = df[SELECTED_FEATURES + ['Diagnosis']].corr()['Diagnosis'].drop('Diagnosis')
target_corr = target_corr.sort_values(key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#ef4444' if v > 0 else '#3b82f6' for v in target_corr]
bars = ax.barh(target_corr.index, target_corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlation with Diagnosis (1 = CKD)')
ax.set_title('Feature-Target Correlation\n(Red = positively correlated with CKD, Blue = negatively)', fontweight='bold')
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
ax.set_xlim([-0.35, 0.35])
plt.tight_layout()
plt.show()

## 10. Symptom Severity — CKD vs No CKD

Three of the 10 selected features are subjective symptom scores (MuscleCramps, Itching, NauseaVomiting).  
This violin plot shows their full distribution shape for each class.


In [ ]:
symptom_cols = ['MuscleCramps', 'Itching', 'NauseaVomiting']
df_melted = df[symptom_cols + ['Diagnosis']].melt(
    id_vars='Diagnosis', var_name='Symptom', value_name='Severity'
)
df_melted['Class'] = df_melted['Diagnosis'].map({1: 'CKD', 0: 'No CKD'})

fig, ax = plt.subplots(figsize=(11, 5))
sns.violinplot(data=df_melted, x='Symptom', y='Severity', hue='Class',
               palette={'CKD': '#ef4444', 'No CKD': '#10b981'},
               split=False, inner='quartile', ax=ax)
ax.set_title('Symptom Severity Distribution by Class', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Severity Score')
plt.tight_layout()
plt.show()

print('Mean symptom severity:')
print(df.groupby('Diagnosis')[symptom_cols].mean().round(2).rename(index={0:'No CKD', 1:'CKD'}))

## 11. All 51 Features — Correlation with Target

This view shows why feature selection was necessary — most features have near-zero  
correlation with the target. Keeping them all adds noise without adding signal.


In [ ]:
all_corr = df.drop(columns=['Diagnosis']).corrwith(df['Diagnosis']).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 14))
colors = ['#ef4444' if v > 0 else '#3b82f6' for v in all_corr[::-1]]
bars = ax.barh(all_corr.index[::-1], all_corr.values[::-1], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)

# Highlight the 10 selected features
selected_set = set(SELECTED_FEATURES)
for bar, feat in zip(bars, all_corr.index[::-1]):
    if feat in selected_set:
        bar.set_edgecolor('#f59e0b')
        bar.set_linewidth(2)

ax.set_xlabel('Correlation with Diagnosis')
ax.set_title('All 51 Features — Correlation with Target\n(Gold outline = selected in final model)',
             fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Features with |corr| > 0.10 : {(all_corr.abs() > 0.10).sum()}')
print(f'Features with |corr| < 0.05 : {(all_corr.abs() < 0.05).sum()} (likely noise)')

## 12. Summary of Findings

| Finding | Detail |
|---|---|
| **Class imbalance** | 91.9% CKD — accuracy is meaningless, use ROC-AUC + Macro F1 |
| **No missing values** | Clean dataset, no imputation needed |
| **Strongest predictors** | SerumCreatinine (r=0.20), GFR (r=-0.18) — clinically expected |
| **GFR threshold** | GFR < 60 → 97.2% CKD rate; GFR ≥ 60 → 87.7% — threshold exists but imperfect |
| **Weak signals overall** | Max feature-target correlation ~0.20 — model must combine features |
| **Symptom features** | MuscleCramps, Itching, NauseaVomiting show moderate separation |
| **No multicollinearity** | No selected feature pairs exceed |r| = 0.5 |
| **41 features dropped** | Near-zero correlation + zero permutation importance (see 02_feature_selection.py) |

---
*Next: `train_model.py` — preprocessing, oversampling, GBM training*
